In [1]:
import os

# --- 0. RESOURCE LIMITATION (Requested: 16 Threads) ---
# Limits the number of threads used by Numpy/Scipy/FakeSpectra to 16
# This prevents the code from hijacking the entire node.
num_cores = 20
os.environ["OMP_NUM_THREADS"] = "20"
os.environ["NUMEXPR_MAX_THREADS"] = "20"
os.environ["MKL_NUM_THREADS"] = "20"
os.environ["VECLIB_MAXIMUM_THREADS"] = "20"

import numpy as np
import h5py
import sys
import pickle
from tqdm import tqdm
import fake_spectra.spectra as spectra
import matplotlib.pyplot as plt

# --- 0. NEW CONFIGURATION ---
def matplotlib_default_config():
    font = {
        'family' : 'serif',
        'weight' : 'normal',
        'size'   : 14
    }
    rcnew = {
        "mathtext.fontset" : "cm", 
        "text.usetex": False,
        'figure.frameon': True,
        'axes.linewidth': 2.,
        "axes.titlesize" : 24,
        "axes.labelsize" : 24,
        "legend.fontsize" : 20,
        'legend.fancybox': True,
        'lines.linewidth': 2.5,
        'xtick.alignment': 'center',
        'xtick.bottom': True,
        'xtick.color': 'black',
        'xtick.direction': 'in',
        'xtick.labelbottom': True,
        'xtick.labelsize': 20,
        'xtick.major.bottom': True,
        'xtick.major.pad': 6.0,
        'xtick.major.size': 10.0,
        'xtick.major.width': 1.5,
        'xtick.minor.bottom': True,
        'xtick.minor.pad': 3.4,
        'xtick.minor.size': 5.0,
        'xtick.minor.visible': True,
        'xtick.minor.width': 1.0,
        'xtick.top': True,
        'ytick.alignment': 'center_baseline',
        'ytick.color': 'black',
        'ytick.direction': 'in',
        'ytick.labelleft': True,
        'ytick.labelsize': 20,
        'ytick.left': True,
        'ytick.major.left': True,
        'ytick.major.pad': 6.0,
        'ytick.major.size': 10.0,
        'ytick.major.width': 1.5,
        'ytick.minor.left': True,
        'ytick.minor.pad': 3.4,
        'ytick.minor.right': True,
        'ytick.minor.size': 5.0,
        'ytick.minor.visible': True,
        'ytick.minor.width': 1.0,
        'ytick.right': True
    }
    return font, rcnew

# Apply Config
%matplotlib widget 
font, rcnew = matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)


In [2]:
# ================= CONFIGURATION =================
SIMULATIONS = [
    {"name": "TNG50-4", "path": "/home/STORAGE/TNG50-4"},
    {"name": "TNG50-3", "path": "/home/STORAGE/TNG50-3"}, 
    {"name": "TNG50-2", "path": "/home/STORAGE/TNG50-2"}
]

SNAP_NUM = 25  # Redshift z=3
MASS_MIN = 10**(12.5) # Solar Masses (Physical)
MASS_MAX = 1e13 # Solar Masses (Physical)
HUBBLE_H = 0.6774 

NUM_RADII = 5
RADI_MULTS = np.logspace(np.log10(0.1), np.log10(3.0), NUM_RADII)

MAX_HALOS = 5 # Increased for testing efficiency
OUTPUT_PKL = "TNG_Resolution_Comparison_biggests.pkl"

print(f"--> CORE LIMIT SET TO: {num_cores}")

--> CORE LIMIT SET TO: 20


In [3]:
# ================= HELPER FUNCTIONS =================

def find_halos_in_mass_bin(base_path, snap_num, m_min, m_max, limit=None):
    """
    Scans TNG group files to find halos within the physical mass range.
    Includes unit conversion fixes for TNG velocities.
    """
    group_dir = os.path.join(base_path, f"groups_{snap_num:03d}")
    f0_path = os.path.join(group_dir, f"fof_subhalo_tab_{snap_num:03d}.0.hdf5")
    
    if not os.path.exists(f0_path): return []

    with h5py.File(f0_path, 'r') as f:
        header = f['Header'].attrs
        num_files = int(header['NumFiles'])
        box_size = header['BoxSize']     
        time = header['Time']            
    
    selected_halos = []
    global_offset = 0
    
    for i in range(num_files):
        g_file = os.path.join(group_dir, f"fof_subhalo_tab_{snap_num:03d}.{i}.hdf5")
        
        with h5py.File(g_file, 'r') as f:
            if 'Group' not in f or 'Group_M_Crit200' not in f['Group']: continue
                
            masses_code = f['Group']['Group_M_Crit200'][:]
            masses_phys = (masses_code * 1e10) / HUBBLE_H
            
            indices = np.where((masses_phys >= m_min) & (masses_phys < m_max))[0]
            
            for idx in indices:
                pos = f['Group']['GroupPos'][idx] 
                # FIX: Velocity in TNG is (km/s)/a. Multiply by 'time' (scale factor), not sqrt(time).
                vel = f['Group']['GroupVel'][idx] * time 
                r200 = f['Group']['Group_R_Crit200'][idx]
                mass = masses_phys[idx]
                
                selected_halos.append({
                    'id': global_offset + idx,
                    'mass': mass,
                    'pos': pos,
                    'vel': vel,
                    'r200': r200,
                    'box': box_size,
                    'time': time
                })
            
            global_offset += int(f['Header'].attrs['Ngroups_ThisFile'])

    if limit is not None and len(selected_halos) > limit:
        np.random.seed(42)
        np.random.shuffle(selected_halos)
        selected_halos = selected_halos[:limit]
        
    return selected_halos

def get_skewers_geometry(center, r200, multiplier):
    """Returns lists of axis and cofm for a specific radius multiplier."""
    impact = r200 * multiplier
    local_cofms = []
    local_axes = []
    offsets = [(impact, 0), (-impact, 0), (0, impact), (0, -impact)]
    
    # Generate 12 skewers (4 per axis)
    # Axis 1 (X)
    for dy, dz in offsets: 
        local_axes.append(1)
        local_cofms.append(center + [0, dy, dz])
    # Axis 2 (Y)
    for dx, dz in offsets: 
        local_axes.append(2)
        local_cofms.append(center + [dx, 0, dz])
    # Axis 3 (Z)
    for dx, dy in offsets: 
        local_axes.append(3)
        local_cofms.append(center + [dx, dy, 0])
        
    return local_axes, local_cofms

def calculate_moments(wave, flux, vel_axis, v_limit=700.0):
    """
    Calcula a EW integrando apenas dentro de uma janela de velocidade +/- v_limit.
    
    Params:
        wave: Array de comprimentos de onda (Angstroms)
        flux: Array de fluxo normalizado (0 a 1)
        vel_axis: Array de velocidades (km/s), onde 0 é o centro do halo
        v_limit: Limite da janela em km/s (ex: 600 significa integrar de -600 a +600)
    """
    # 1. Criar uma máscara booleana para a região de interesse
    # Queremos apenas onde a velocidade está entre -v_limit e +v_limit
    mask = (vel_axis >= -v_limit) & (vel_axis <= v_limit)
    
    # Se a máscara estiver vazia (erro de limites), retorna 0
    if not np.any(mask):
        return 0.0

    # 2. Recortar os arrays
    flux_window = flux[mask]
    wave_window = wave[mask]
    
    # 3. Calcular a absorção (1 - F)
    absorption = 1.0 - flux_window
    
    # 4. Integrar (Trapz) apenas na janela
    # O resultado será em Angstroms, pois x=wave_window
    eq_width = np.trapz(absorption, x=wave_window)
    
    return eq_width

In [4]:
# ================= MAIN PIPELINE (OPTIMIZED & FULL DATA STORAGE) =================

master_data = {}

for sim in tqdm(SIMULATIONS, desc="Simulations"):
    sim_name = sim['name']
    
    print(f"\nScanning {sim_name}...")
    halos = find_halos_in_mass_bin(sim['path'], SNAP_NUM, MASS_MIN, MASS_MAX, limit=MAX_HALOS)
    print(f"   > Found {len(halos)} halos.")
    
    sim_results = []
    
    # Processing Halos
    for halo in tqdm(halos, desc=f"Processing {sim_name}", leave=False):
        
        # We store the Halo Mass at the top level. 
        # When plotting, we broadcast this single mass value to all 12 skewers.
        halo_result = {
            'id': halo['id'], 
            'mass': halo['mass'], 
            'r200': halo['r200'], 
            'radii_data': []
        }
        
        # --- OPTIMIZATION STEP 1: PRE-CALCULATE ALL GEOMETRIES ---
        # Instead of calling Spectra 5 times, we prepare one big list of 60 skewers.
        
        all_axes = []
        all_cofms = []
        
        # Keep track of which indices belong to which radius multiplier
        # structure: { r_mult: [start_index, end_index] }
        slice_map = {} 
        current_idx = 0
        
        for r_mult in RADI_MULTS:
            axes, cofms = get_skewers_geometry(halo['pos'], halo['r200'], r_mult)
            num_skewers = len(axes) # Should be 12
            
            all_axes.extend(axes)
            all_cofms.extend(cofms)
            
            slice_map[r_mult] = (current_idx, current_idx + num_skewers)
            current_idx += num_skewers
            
        # Convert to numpy arrays for Fake Spectra
        all_axes = np.array(all_axes)
        all_cofms = np.array(all_cofms)
        
        # --- OPTIMIZATION STEP 2: RUN SPECTRA ONCE ---
        save_file = f"temp_{sim_name}_{halo['id']}_ALL.hdf5"
        if os.path.exists(save_file): os.remove(save_file)
        
        try:
            # This is the heavy lifting - done only once per halo now!
            ss = spectra.Spectra(SNAP_NUM, sim['path'], cofm=all_cofms, axis=all_axes, 
                                 res=10, savefile=save_file, reload_file=True)
            
            # Get data for all 60 skewers at once
            tau_all = ss.get_tau("H", 1, 1215)
            flux_all = np.exp(-tau_all)
            nbins = ss.nbins
            vmax = ss.vmax
            
            # Generate Velocity/Wave axis (same for all)
            vel_axis = np.linspace(-vmax/2, vmax/2, nbins)
            wave_axis = 1215.67 * (1 + vel_axis / 299792.458)

            # --- OPTIMIZATION STEP 3: POST-PROCESS BY BATCH ---
            
            for r_mult in RADI_MULTS:
                # Extract the 12 skewers corresponding to this radius
                start, end = slice_map[r_mult]
                flux_subset = flux_all[start:end]
                axes_subset = all_axes[start:end]
                
                flux_centered_list = []
                
                # Vectorized centering 
                for k in range(12):
                    axis_idx = axes_subset[k] - 1
                    
                    # Hubble Flow + Peculiar Velocity Correction
                    v_hub = (halo['pos'][axis_idx] / halo['box']) * vmax
                    v_pec = halo['vel'][axis_idx] 
                    v_total = v_hub + v_pec
                    
                    pix_halo = int((v_total / vmax) * nbins) % nbins
                    pix_center = nbins // 2
                    shift = pix_center - pix_halo
                    
                    # Roll the flux to center the halo
                    f_cent = np.roll(flux_subset[k], shift)
                    flux_centered_list.append(f_cent)
                
                flux_centered_matrix = np.array(flux_centered_list)
                
                # Calculate Moments (Individual Values)
                # Ensure calculate_moments accepts v_limit and vel_axis as updated previously
                ew_values = [calculate_moments(wave_axis, f, vel_axis, v_limit=700) for f in flux_centered_matrix]
                
                # Convert to numpy array for storage efficiency
                ew_values = np.array(ew_values, dtype=np.float32)

                # Store EVERYTHING (Individual and Aggregate)
                halo_result['radii_data'].append({
                    'r_mult': r_mult,
                    'wave': wave_axis.astype(np.float32),
                    'flux_matrix': flux_centered_matrix.astype(np.float32),
                    
                    # Aggregates (Quick look)
                    'ew_mean': np.mean(ew_values),
                    'ew_std': np.std(ew_values),
                    
                    # INDIVIDUAL VALUES (Crucial for scatter plots/statistics)
                    'ew_individual': ew_values 
                })

        except Exception as e:
            print(f"Error processing halo {halo['id']}: {e}")
        finally:
            if os.path.exists(save_file): os.remove(save_file)
            
        sim_results.append(halo_result)

    master_data[sim_name] = sim_results

# ================= SAVE TO DISK =================
print(f"\n💾 Saving statistics to {OUTPUT_PKL}...")
with open(OUTPUT_PKL, 'wb') as f:
    pickle.dump(master_data, f)
print(f"✅ Processing Complete. Data Size: {os.path.getsize(OUTPUT_PKL) / 1e6:.2f} MB")

Simulations:   0%|          | 0/3 [00:00<?, ?it/s]


Scanning TNG50-4...
   > Found 5 halos.


Reloading from snapshot (will save to:  /home/STORAGE/TNG50-4/snapdir_025/temp_TNG50-4_0_ALL.hdf5  )
60  sightlines. resolution:  10.003904128669364  z= 3.008131071630377
rank = 0 | Interpolation 9.1 percent done
rank = 0 | Interpolation 18.2 percent done
rank = 0 | Interpolation 27.3 percent done
rank = 0 | Interpolation 36.4 percent done
rank = 0 | Interpolation 45.5 percent done
rank = 0 | Interpolation 54.5 percent done
rank = 0 | Interpolation 63.6 percent done
rank = 0 | Interpolation 72.7 percent done
rank = 0 | Interpolation 81.8 percent done
rank = 0 | Interpolation 90.9 percent done


Reloading from snapshot (will save to:  /home/STORAGE/TNG50-4/snapdir_025/temp_TNG50-4_1_ALL.hdf5  )
60  sightlines. resolution:  10.003904128669364  z= 3.008131071630377
rank = 0 | Interpolation 9.1 percent done
rank = 0 | Interpolation 18.2 percent done
rank = 0 | Interpolation 27.3 percent done
rank = 0 | Interpolation 36.4 percent done
rank = 0 | Interpolation 45.5 percent done
rank = 0 | Interpolation 54.5 percent done
rank = 0 | Interpolation 63.6 percent done
rank = 0 | Interpolation 72.7 percent done
rank = 0 | Interpolation 81.8 percent done
rank = 0 | Interpolation 90.9 percent done


Reloading from snapshot (will save to:  /home/STORAGE/TNG50-4/snapdir_025/temp_TNG50-4_2_ALL.hdf5  )
60  sightlines. resolution:  10.003904128669364  z= 3.008131071630377
rank = 0 | Interpolation 9.1 percent done
rank = 0 | Interpolation 18.2 percent done
rank = 0 | Interpolation 27.3 percent done
rank = 0 | Interpolation 36.4 percent done
rank = 0 | Interpolation 45.5 percent done
rank = 0 | Interpolation 54.5 percent done
rank = 0 | Interpolation 63.6 percent done
rank = 0 | Interpolation 72.7 percent done
rank = 0 | Interpolation 81.8 percent done
rank = 0 | Interpolation 90.9 percent done


Reloading from snapshot (will save to:  /home/STORAGE/TNG50-4/snapdir_025/temp_TNG50-4_5_ALL.hdf5  )
60  sightlines. resolution:  10.003904128669364  z= 3.008131071630377
rank = 0 | Interpolation 9.1 percent done
rank = 0 | Interpolation 18.2 percent done
rank = 0 | Interpolation 27.3 percent done
rank = 0 | Interpolation 36.4 percent done
rank = 0 | Interpolation 45.5 percent done
rank = 0 | Interpolation 54.5 percent done
rank = 0 | Interpolation 63.6 percent done
rank = 0 | Interpolation 72.7 percent done
rank = 0 | Interpolation 81.8 percent done
rank = 0 | Interpolation 90.9 percent done


... (13 outputs left out)

Reloading from snapshot (will save to:  /home/STORAGE/TNG50-2/snapdir_025/temp_TNG50-2_5_ALL.hdf5  )
60  sightlines. resolution:  10.003904128669364  z= 3.008131071630377
rank = 0 | Interpolation 0.8 percent done
rank = 0 | Interpolation 1.6 percent done
rank = 0 | Interpolation 2.3 percent done
rank = 0 | Interpolation 3.1 percent done
rank = 0 | Interpolation 3.9 percent done
rank = 0 | Interpolation 4.7 percent done
rank = 0 | Interpolation 5.5 percent done
rank = 0 | Interpolation 6.2 percent done
rank = 0 | Interpolation 7.0 percent done
rank = 0 | Interpolation 7.8 percent done
rank = 0 | Interpolation 8.6 percent done
rank = 0 | Interpolation 9.4 percent done
rank = 0 | Interpolation 10.2 percent done
rank = 0 | Interpolation 10.9 percent done
rank = 0 | Interpolation 11.7 percent done
rank = 0 | Interpolation 12.5 percent done
rank = 0 | Interpolation 13.3 percent done
rank = 0 | Interpolation 14.1 percent done
rank = 0 | Interpolation 14.8 percent done
rank = 0 | Interpolation

Reloading from snapshot (will save to:  /home/STORAGE/TNG50-2/snapdir_025/temp_TNG50-2_7_ALL.hdf5  )
60  sightlines. resolution:  10.003904128669364  z= 3.008131071630377
rank = 0 | Interpolation 0.8 percent done
rank = 0 | Interpolation 1.6 percent done
rank = 0 | Interpolation 2.3 percent done
rank = 0 | Interpolation 3.1 percent done
rank = 0 | Interpolation 3.9 percent done
rank = 0 | Interpolation 4.7 percent done
rank = 0 | Interpolation 5.5 percent done
rank = 0 | Interpolation 6.2 percent done
rank = 0 | Interpolation 7.0 percent done
rank = 0 | Interpolation 7.8 percent done
rank = 0 | Interpolation 8.6 percent done
rank = 0 | Interpolation 9.4 percent done
rank = 0 | Interpolation 10.2 percent done
rank = 0 | Interpolation 10.9 percent done
rank = 0 | Interpolation 11.7 percent done
rank = 0 | Interpolation 12.5 percent done
rank = 0 | Interpolation 13.3 percent done
rank = 0 | Interpolation 14.1 percent done
rank = 0 | Interpolation 14.8 percent done
rank = 0 | Interpolation

Simulations: 100%|██████████| 3/3 [07:04<00:00, 141.50s/it]


💾 Saving statistics to TNG_Resolution_Comparison_biggests.pkl...
✅ Processing Complete. Data Size: 1.56 MB


In [ ]:
import pickle
import numpy as np
import os

# Nome do arquivo que salvamos
INPUT_PKL = "TNG_Resolution_Comparison_12_13.pkl"

def inspect_pickle(path):
    if not os.path.exists(path):
        print(f"❌ Arquivo não encontrado: {path}")
        return

    print(f"📂 Carregando: {path}...\n")
    
    with open(path, 'rb') as f:
        data = pickle.load(f)
    
    # 1. Nível Superior: Simulações
    print(f"🔑 Simulações encontradas: {list(data.keys())}")
    
    for sim_name, halos in data.items():
        print(f"\n" + "="*40)
        print(f"🔹 Simulação: {sim_name}")
        print(f"   ↳ Quantidade de Halos salvos: {len(halos)}")
        
        if len(halos) == 0:
            print("   (Vazio)")
            continue
            
        # 2. Pegar o PRIMEIRO halo para ver a estrutura
        first_halo = halos[0]
        print(f"\n   🔎 Inspecionando o 1º Halo (ID {first_halo['id']}):")
        
        # Propriedades do Halo
        print(f"      • Massa (M200): {first_halo['mass']:.2e} Msun")
        print(f"      • Raio (R200):  {first_halo['r200']:.2f} ckpc/h")
        
        # 3. Inspecionar Dados dos Raios (Skewers)
        radii_list = first_halo['radii_data']
        print(f"      • Entradas de Raios (Impact Parameters): {len(radii_list)}")
        
        if len(radii_list) > 0:
            # Pegar o primeiro raio (ex: 0.1 R200)
            r_data = radii_list[0]
            print(f"\n      Testando o 1º Raio (b = {r_data['r_mult']:.2f} R200):")
            print(f"      --------------------------------------------------")
            
            # Listar todas as chaves salvas neste dicionário
            for key, value in r_data.items():
                if isinstance(value, np.ndarray):
                    print(f"      👉 '{key}': Array NumPy com shape {value.shape}")
                    
                    # Se for o array de EW individual que acabamos de criar
                    if key == 'ew_individual':
                        print(f"          Values: {value}")
                        
                elif isinstance(value, (float, int)):
                    print(f"      👉 '{key}': {value:.4f}")
                else:
                    print(f"      👉 '{key}': Tipo {type(value)}")

# Rodar a função
inspect_pickle(INPUT_PKL)

In [ ]:
# --- CONFIGURATION ---
INPUT_PKL = "TNG_Resolution_Comparison_12_13.pkl"
LAMBDA_LYALPHA = 1215.67 
LIGHT_SPEED = 299792.458 

# How many thin lines to plot in the background?
# Plotting 12,000 lines (1000 halos * 12 skewers) is too heavy.
# We will calculate stats on ALL, but verify visually with a sample.
VISUAL_SAMPLE_SIZE = 100 

SIM_CONFIG = {
    "TNG50-4": {"color": "#1f77b4", "label": "TNG50-4"},
    "TNG50-3": {"color": "#ff7f0e", "label": "TNG50-3"},
    "TNG50-2": {"color": "#2ca02c", "label": "TNG50-2"}
}

def apply_plot_style():
    plt.style.use('default')
    plt.rc('font', family='serif', size=14)
    plt.rcParams['xtick.direction'] = 'in'
    plt.rcParams['ytick.direction'] = 'in'

# --- LOAD DATA ---
print(f"📂 Loading data from {INPUT_PKL}...")
with open(INPUT_PKL, 'rb') as f:
    master_data = pickle.load(f)

apply_plot_style()

# We assume 5 radii bins based on your generation code
num_radii = 5 
fig, axes = plt.subplots(nrows=num_radii, ncols=1, figsize=(10, 20), sharex=True)
plt.subplots_adjust(hspace=0.05)

print("Processing stack for all halos...")

# --- MAIN LOOP: PER RADIAL BIN ---
for i, ax in enumerate(axes):
    
    # --- INNER LOOP: PER SIMULATION ---
    for sim_name, style in SIM_CONFIG.items():
        if sim_name not in master_data: continue
            
        halos = master_data[sim_name]
        if not halos: continue
        
        # 1. GATHER DATA (The Stacking Step)
        # We need to extract the flux matrix for radius 'i' from EVERY halo.
        # radius_fluxes will be a list of arrays, each shape (12, nbins)
        radius_fluxes = []
        
        # We need the wave axis from at least one halo to convert to velocity
        # (Assuming all skewers have same binning/resolution)
        sample_wave = halos[0]['radii_data'][i]['wave']
        
        for halo in halos:
            # Check if this halo has data for this radius (sanity check)
            if i < len(halo['radii_data']):
                f_matrix = halo['radii_data'][i]['flux_matrix']
                radius_fluxes.append(f_matrix)
        
        # 2. CREATE GIANT MATRIX
        # Stack vertically. 
        # If 1000 halos, shape becomes (12000, nbins)
        if len(radius_fluxes) > 0:
            big_flux_matrix = np.vstack(radius_fluxes)
        else:
            continue
            
        # Convert Wavelength to Velocity
        vel_axis = LIGHT_SPEED * (sample_wave / LAMBDA_LYALPHA - 1)

        # 3. CALCULATE GLOBAL STATISTICS
        # These are statistically robust across the entire population
        mean_flux = np.nanmean(big_flux_matrix, axis=0)
        std_flux = np.nanstd(big_flux_matrix, axis=0)
        
        # 4. PLOT VISUAL SAMPLE (Thin Lines)
        # We randomly select indices to plot so we don't crash the viewer
        total_skewers = big_flux_matrix.shape[0]
        if total_skewers > VISUAL_SAMPLE_SIZE:
            indices = np.random.choice(total_skewers, VISUAL_SAMPLE_SIZE, replace=False)
            sample_matrix = big_flux_matrix[indices]
        else:
            sample_matrix = big_flux_matrix

        # Plot the sample thin lines
        for flux in sample_matrix:
            ax.plot(vel_axis, flux, color=style['color'], 
                    alpha=0.05, linewidth=0.5, zorder=1) # Very transparent
            
        # 5. PLOT GLOBAL MEAN AND STD
        label_text = f"{style['label']} (N={len(halos)})" if i == 0 else ""
        
        ax.plot(vel_axis, mean_flux, color=style['color'], 
                linewidth=2.5, label=label_text, zorder=3)
        
        ax.fill_between(vel_axis, mean_flux - std_flux, mean_flux + std_flux,
                        color=style['color'], alpha=0.2, zorder=2)
        
        # Capture impact parameter label from first halo
        r_mult = halos[0]['radii_data'][i]['r_mult']

    # --- FORMATTING ---
    ax.text(0.02, 0.05, f"$b = {r_mult:.1f} \ R_{{200}}$", transform=ax.transAxes,
            fontsize=16, fontweight='bold', bbox=dict(facecolor='white', alpha=0.9))
    
    ax.set_ylabel(r"Normalized Flux ($F$)")
    ax.set_ylim(-0.05, 1.1)
    ax.grid(True, linestyle=':', alpha=0.4)
    ax.axhline(1, color='k', ls='--', alpha=0.5)
    ax.axhline(0, color='k', ls='-', lw=0.5, alpha=0.5)

axes[-1].set_xlabel(r"Velocity ($v$ [km/s])")
axes[-1].set_xlim(-2000, 2000) # Zoom in to relevant region
axes[0].legend(loc="lower right")
axes[0].set_title(r"Stacked Lyman-$\alpha$ Profiles (Mean of all Halos)")
plt.savefig("Stacked_LyA_Profiles.png", dpi=300, bbox_inches="tight")

plt.show()

### Now the plot of the equivalent width with respect to the Halo Mass: 

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# --- CONFIGURAÇÃO SIMPLES ---
INPUT_PKL = "TNG_Resolution_Comparison.pkl"

# Cores das simulações
sim_colors = {
    "TNG50-4": "C0", # Azul
    "TNG50-3": "C1", # Laranja
    "TNG50-2": "C2"  # Verde
}

# Carregar dados
with open(INPUT_PKL, 'rb') as f:
    master_data = pickle.load(f)

# Configurar Bins de Massa (Log10)
bins = np.linspace(10.0, 11.0, 9) 
bin_centers = (bins[:-1] + bins[1:]) / 2
bin_width = bins[1] - bins[0]

# Criar Figura
fig, axes = plt.subplots(nrows=5, ncols=1, figsize=(10, 20), sharex=True)
plt.subplots_adjust(hspace=0)

# Loop pelos 5 painéis
for i, ax in enumerate(axes):
    
    for sim_name, color in sim_colors.items():
        if sim_name not in master_data: continue
        
        halos = master_data[sim_name]
        
        # 1. Coletar dados
        all_mass = []
        all_ew = []
        
        for halo in halos:
            if i < len(halo['radii_data']):
                r_data = halo['radii_data'][i]
                if 'ew_individual' in r_data:
                    vals = r_data['ew_individual']
                    m_val = halo['mass']
                    all_mass.extend([m_val] * len(vals))
                    all_ew.extend(vals)
        
        all_mass = np.array(all_mass)
        all_ew = np.array(all_ew)
        
        # Filtro e Log
        mask = (all_ew > 0.01)
        log_mass = np.log10(all_mass[mask])
        log_ew = np.log10(all_ew[mask])
        
        # 2. Dados por bin
        violin_data = []
        valid_positions = []
        
        for b in range(len(bin_centers)):
            mask_bin = (log_mass >= bins[b]) & (log_mass < bins[b+1])
            data_in_bin = log_ew[mask_bin]
            
            if len(data_in_bin) > 5:
                violin_data.append(data_in_bin)
                valid_positions.append(bin_centers[b])
        
        # 3. Plotar Violino e Linhas Conectadas
        if len(violin_data) > 0:
            # Violino (Contorno apenas)
            parts = ax.violinplot(violin_data, positions=valid_positions, 
                                  widths=bin_width*0.8,
                                  showextrema=False, bw_method=0.2)
            
            for pc in parts['bodies']:
                pc.set_facecolor('none')
                pc.set_edgecolor(color)
                pc.set_linewidth(2)
                pc.set_alpha(0.7)
            
            # --- AQUI ESTÁ A MUDANÇA ---
            # Calcular medianas
            medians = [np.median(d) for d in violin_data]
            
            # Plotar bolinhas conectadas por linha ('-o')
            ax.plot(valid_positions, medians, '-o', color=color, 
                    linewidth=2, markersize=6, label=sim_name if i==0 else "")

    # Texto do parametro de impacto
    if len(halos) > 0:
        r_mult = halos[0]['radii_data'][i]['r_mult']
        ax.text(0.02, 0.85, f"$b = {r_mult:.1f} R_{{200}}$", transform=ax.transAxes)
    
    # Linha de referência em 0 (1 Angstrom)
    ax.axhline(0, color='k', linestyle='--', alpha=0.3)
    ax.set_ylim(-2.0, 1.2)
    ax.set_ylabel(r"$\log_{10}(\mathrm{EW} / \AA)$")

# Eixo X
axes[-1].set_xlabel(r"$\log_{10}(M_{200} / M_\odot)$")
axes[-1].set_xlim(10.0, 11.0)

# Legenda (agora usa os labels definidos no ax.plot)
axes[0].legend(loc="lower right", ncol=3)

plt.show()